# Training hyperparameter optimization: PyTorch & Optuna

**Optuna Dashboard:** To monitor optimization progress in real-time, start the Optuna dashboard in a terminal:

```text
optuna-dashboard sqlite:///data/pytorch/training_optimization.db
```

Then open http://localhost:8080 in your browser.

## 1. Notebook setup

### 1.1. Imports

In [ ]:
# Standard library imports
import pickle

# Third party imports
import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets

# Package imports
from image_classification_tools.pytorch import DataPipeline
import image_classification_tools.pytorch.evaluation as eval_utils
import image_classification_tools.pytorch.hyperparameter_optimization as optimization
import image_classification_tools.pytorch.plotting as plots
import image_classification_tools.pytorch.training as training

# Local imports
import configuration as config
import helper_functions as hf

### 1.2. Run configuration

In [ ]:
# Model control
rerun_optuna_study = True    # Set to False to load existing Optuna study from disk
retrain_model = True         # Set to False to load existing final model
model_source = 'huggingface' # 'local' or 'huggingface'

# Optuna study configuration
study_name = 'cnn_training_optimization'
study_storage = 'sqlite:///../data/pytorch/training_optimization.db'

# Parallel GPU configuration
n_parallel_workers = torch.cuda.device_count() if torch.cuda.is_available() else 1

### 1.3. Fixed hyperparameters

In [ ]:
# Data loading
preload_device = 'gpu' if torch.cuda.is_available() else 'cpu'

# Optuna optimization settings
n_trials = 100
epochs_per_trial = 50
pruner_warmup_steps = 5
trial_early_stopping_patience = 10

# Final model training settings
epochs = 500
early_stopping_patience = 15
print_every = 20

## 2. Load best architecture from architecture optimization

We'll load the best architecture hyperparameters from notebook 04.

In [ ]:
# Load architecture optimization study from notebook 04
architecture_study = optuna.load_study(
    study_name='cnn_architecture_optimization',
    storage='sqlite:///../data/pytorch/cnn_optimization.db'
)

# Get best architecture parameters
arch_params = architecture_study.best_trial.params

print('Best architecture hyperparameters from notebook 04:\n')
for key, value in arch_params.items():
    print(f'  {key}: {value}')

print(f'\nBest validation accuracy: {architecture_study.best_value:.2f}%')

## 3. Plot sample images

In [ ]:
# Get a sample dataset for visualization
sample_dataset = datasets.CIFAR10(
    root=config.DATA_DIR,
    train=True,
    transform=config.RGB_TRANSFORM
)

# Plot first 10 images from the training dataset
fig, axes = plots.plot_sample_images(sample_dataset, config.CLASS_NAMES)
plt.show()

## 4. Optuna training hyperparameter optimization

In [ ]:
# Define model path
model_path = config.MODELS_DIR / 'training_optimized_cnn.pth'

# Load existing model if requested
if not retrain_model:

    model = hf.load_model_from_source(
        model_path=model_path,
        model_name='training_optimized_cnn.pth',
        model_source=model_source,
        device=config.DEVICE
    )
    
    # Try to load training history
    history = hf.load_history_from_source(
        model_path=model_path,
        model_name='training_optimized_cnn.pth',
        model_source=model_source
    )
    
    if model is None:
        print('Pretrained model not available, proceeding with training...')
        retrain_model = True

else:
    print('Training new model...')
    history = None

### 4.1. Define training hyperparameter search space

In [ ]:
# Define training hyperparameter search space
search_space = {
    'batch_size': [64, 128, 256],
    'learning_rate': (1e-5, 1e-2, 'log'),
    'optimizer': ['Adam', 'AdamW', 'SGD'],
    'weight_decay': (1e-6, 1e-3, 'log')
}

print('Training hyperparameter search space:\n')
for key, value in search_space.items():
    print(f'  {key}: {value}')

### 4.2. Create CNN model factory (using best architecture)

In [ ]:
def create_cnn(trial, num_classes, in_channels):
    '''Create a CNN with fixed architecture and trial-sampled training hyperparameters.
    
    Uses the best architecture from notebook 04, but samples training
    hyperparameters like optimizer, learning rate, etc. from the trial.
    
    Args:
        trial: Optuna trial object for suggesting hyperparameters
        num_classes: Number of output classes
        in_channels: Number of input channels (3 for RGB, 1 for grayscale)
    
    Returns:
        nn.Sequential model
    '''
    
    # Use fixed architecture parameters from notebook 04
    n_conv_blocks = arch_params['n_conv_blocks']
    initial_filters = arch_params['initial_filters']
    n_fc_layers = arch_params['n_fc_layers']
    conv_dropout_rate = arch_params['conv_dropout_rate']
    fc_dropout_rate = arch_params['fc_dropout_rate']
    pool_frequency = arch_params.get('pool_frequency', 2)
    filter_double_frequency = arch_params.get('filter_double_frequency', 2)
    
    layers = []
    current_channels = in_channels
    spatial_size = 32  # CIFAR-10 input size
    
    # Convolutional blocks
    for block_idx in range(n_conv_blocks):
        # Double filters every N blocks
        out_channels = initial_filters * (2 ** (block_idx // filter_double_frequency))
        
        # First conv in block
        layers.append(nn.Conv2d(current_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU())
        
        # Second conv in block
        layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU())
        
        # Pool every N blocks
        should_pool = (block_idx + 1) % pool_frequency == 0
        if should_pool and spatial_size > 1:
            layers.append(nn.MaxPool2d(2, 2))
            spatial_size //= 2
        
        layers.append(nn.Dropout(conv_dropout_rate))
        current_channels = out_channels
    
    # Classifier with adaptive pooling
    layers.append(nn.AdaptiveAvgPool2d((1, 1)))
    layers.append(nn.Flatten())
    
    # Generate FC layer sizes
    fc_sizes = []
    current_fc_size = current_channels // 2
    for _ in range(n_fc_layers):
        fc_sizes.append(max(32, current_fc_size))
        current_fc_size //= 2
    
    # Add FC layers
    in_features = current_channels
    for fc_size in fc_sizes:
        layers.append(nn.Linear(in_features, fc_size))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(fc_dropout_rate))
        in_features = fc_size
    
    # Output layer
    layers.append(nn.Linear(in_features, num_classes))
    
    return nn.Sequential(*layers)

### 4.3. Run Optuna optimization study

In [ ]:
%%time

if rerun_optuna_study:
    
    # Display parallel worker configuration
    print(f'Running optimization with {n_parallel_workers} parallel workers')

    if torch.cuda.is_available():
        print(f'Available GPUs: {torch.cuda.device_count()}')

        for i in range(torch.cuda.device_count()):
            print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
    
    # Create objective function
    objective = optimization.create_objective(
        model_factory=create_cnn,
        data_source=datasets.CIFAR10,
        data_dir=config.DATA_DIR,
        train_transform=config.RGB_TRANSFORM,
        eval_transform=config.RGB_TRANSFORM,
        n_epochs=epochs_per_trial,
        num_classes=len(config.CLASS_NAMES),
        in_channels=3,
        val_size=config.VAL_SIZE,
        search_space=search_space,
        early_stopping_patience=trial_early_stopping_patience
    )
    
    # Create Optuna study with pruning and database storage
    study = optuna.create_study(
        direction='maximize',
        study_name=study_name,
        storage=study_storage,
        load_if_exists=True,
        pruner=optuna.pruners.MedianPruner(
            n_warmup_steps=pruner_warmup_steps
        )
    )
    
    # Run optimization
    study.optimize(
        objective, 
        n_trials=n_trials, 
        n_jobs=n_parallel_workers,
        catch=(optimization.TrialFailedError,)
    )
    
    print(f'\nBest trial: {study.best_trial.number}')
    print(f'Best validation accuracy: {study.best_value:.2f}%')

else:

    # Load existing study
    study = optuna.load_study(
        study_name=study_name,
        storage=study_storage
    )
    print(f'Study loaded from: {study_storage}')
    print(f'Best validation accuracy: {study.best_value:.2f}%')

### 4.4. Visualize optimization results

In [ ]:
# Print best hyperparameters
print('Best hyperparameters:')
for key, value in study.best_params.items():
    print(f'  {key}: {value}')

# Plot optimization history
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

# Plot parameter importances
fig = optuna.visualization.plot_param_importances(study)
fig.show()

# Plot parallel coordinate
fig = optuna.visualization.plot_parallel_coordinate(study)
fig.show()

## 5. Train final model with best training hyperparameters

### 5.1. Load winning hyperparameters

In [ ]:
if retrain_model:

    # Get best hyperparameters from Optuna study
    best_params = study.best_params

    # Display best hyperparameters
    print('Training with best hyperparameters:')
    for key, value in best_params.items():
        print(f'  {key}: {value}')

### 5.2. Re-create dataloader with train/test split

In [ ]:
if retrain_model:

    # Create dataloaders with train/test split for final training
    dataloaders = data.DataPipeline(
        data_source=datasets.CIFAR10,
        data_dir=config.DATA_DIR,
        batch_size=best_params['batch_size'],
        train_transform=config.RGB_TRANSFORM,
        eval_transform=config.RGB_TRANSFORM,
        split='train/test',
        test_size=config.TEST_SIZE,
        random_state=config.RANDOM_STATE,
        preload_to_device=config.DEVICE,
        num_workers=config.NUM_WORKERS,
        verbose=True
    )

    train_loader = dataloaders.get('train')
    test_loader = dataloaders.get('test')

### 5.3. Create optimized model

In [ ]:
if retrain_model:

    # Create mock trial with best hyperparameters
    mock_trial = optimization.MockTrial(best_params)

    # Create model using the model factory
    model = create_cnn(
        trial=mock_trial,
        num_classes=len(config.CLASS_NAMES),
        in_channels=3
    )
    model = model.to(config.DEVICE)

    # Print model summary
    print(f'Model created with {sum(p.numel() for p in model.parameters()):,} parameters')
    print(f'Device: {config.DEVICE}')

### 5.4. Train model

In [ ]:
%%time

if retrain_model:
    
    # Train the model
    model, history = training.train_model(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        n_epochs=config.NUM_EPOCHS,
        learning_rate=best_params['learning_rate'],
        device=config.DEVICE,
        early_stopping_patience=config.EARLY_STOPPING_PATIENCE,
        optimizer=best_params['optimizer'],
        weight_decay=best_params['weight_decay'],
        verbose=True
    )

    print(f'\nFinal training accuracy: {history["train_acc"][-1]:.2f}%')
    print(f'Final test accuracy: {history["test_acc"][-1]:.2f}%')

### 5.5. Plot learning curves

In [ ]:
if history is not None:
    fig, axes = plots.plot_learning_curves(history)
    plt.show()

## 6. Evaluate on test set

### 6.1. Test accuracy

In [ ]:
# Recreate test dataloader if not already created
if not retrain_model:
    test_loader = data.DataPipeline(
        data_source=datasets.CIFAR10,
        data_dir=config.DATA_DIR,
        batch_size=128,
        train_transform=config.RGB_TRANSFORM,
        eval_transform=config.RGB_TRANSFORM,
        split='train/test',
        test_size=config.TEST_SIZE,
        random_state=config.RANDOM_STATE,
        preload_to_device=config.DEVICE,
        num_workers=config.NUM_WORKERS,
        verbose=False
    ).get('test')

# Evaluate
test_acc, y_true, y_pred, y_probs = evaluation.evaluate_model(
    model=model,
    test_loader=test_loader,
    device=config.DEVICE,
    return_predictions=True
)

print(f'\nTest accuracy: {test_acc:.2f}%')

### 6.2. Per-class accuracy

In [ ]:
# Plot per-class accuracy
fig, ax = plots.plot_per_class_accuracy(
    y_true=y_true,
    y_pred=y_pred,
    class_names=config.CLASS_NAMES
)
plt.show()

### 6.3. Confusion matrix

In [ ]:
# Plot confusion matrix
fig, ax = plots.plot_confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    class_names=config.CLASS_NAMES
)
plt.show()

### 6.4. Probability distributions

In [ ]:
# Plot prediction probability distributions
fig, axes = plots.plot_probability_distributions(
    y_true=y_true,
    y_probs=y_probs
)
plt.show()

### 6.5. Evaluation curves

In [ ]:
# Plot ROC curves
fig, ax = plots.plot_roc_curves(
    y_true=y_true,
    y_probs=y_probs,
    class_names=config.CLASS_NAMES
)
plt.show()

# Plot precision-recall curves
fig, ax = plots.plot_precision_recall_curves(
    y_true=y_true,
    y_probs=y_probs,
    class_names=config.CLASS_NAMES
)
plt.show()

## 7. Save model

In [ ]:
if retrain_model:
    
    # Save the trained model with history
    torch.save({
        'model_state_dict': model.state_dict(),
        'history': history,
        'best_params': best_params,
        'arch_params': arch_params,
        'test_accuracy': test_acc
    }, model_path)
    
    print(f'Model saved to {model_path}')
    print(f'Test accuracy: {test_acc:.2f}%')

## 8. Save results to performance tracker

In [ ]:
# Save performance results
results_file = config.DATA_DIR / 'pytorch' / 'performance_results' / 'training_optimized_cnn.json'

if retrain_model:
    
    results = {
        'model': 'training_optimized_cnn',
        'test_accuracy': test_acc,
        'final_train_accuracy': history['train_acc'][-1],
        'final_test_accuracy': history['test_acc'][-1],
        'best_epoch': int(np.argmax(history['test_acc'])),
        'epochs_trained': len(history['train_loss']),
        'architecture_params': arch_params,
        'training_params': best_params,
        'timestamp': str(pd.Timestamp.now())
    }
    
    # Create directory if it doesn't exist
    results_file.parent.mkdir(parents=True, exist_ok=True)
    
    # Save
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f'Results saved to {results_file}')

else:
    print('Skipping save (retrain_model=False)')

---

**Note:** To visualize the Optuna study in real-time, run:

```bash
optuna-dashboard sqlite:///data/pytorch/performance_results/training_optuna.db
```

Then open the URL shown in your browser.